# TeenPattiBench — batch evaluation, 9 models × 2 eval sets

For each model, in order: load it once → run the **v1 eval** → run the **shuffled eval**
→ score both → free the GPU and delete the downloaded weights → next model. At the
end, one zip with every prediction file, every metrics report and a summary table.

Per-item method is the standard Phase 4 one: chat template with the system prompt as
a `system` turn, greedy decoding, `max_new_tokens=16`, batch 8, left padding, thinking
off, first action word in the completion (not restricted to legal actions), output
as a metadata header plus `{"id", "answer"}` lines.

### Before you run
1. **Runtime → Change runtime type → A100.** Eight of the nine models fit in its
   40 GB. **Qwen3-30B-A3B needs ~61 GB and is skipped automatically** on an A100,
   with the reason recorded in the log and summary. On an H100 (80 GB) it runs with
   no changes.
2. **Accept the licence** for the gated models, logged in to Hugging Face:
   - <https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct>
   - <https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3>
   - <https://huggingface.co/google/gemma-3-4b-it>
   - <https://huggingface.co/google/gemma-3-12b-it>

   Cell 6 checks access to every model before anything downloads, so a missing licence
   shows up in the first minute rather than two hours in.
3. **🔑 Secrets:** add `HF_TOKEN` (a *read* token) and enable notebook access.
4. **📁 Files:** upload these four into `/content`:
   `teenpattibench_eval.jsonl`, `teenpattibench_eval_shuffled.jsonl`,
   `teenpattibench_system_prompt.txt`, `metrics.py`
5. **Runtime → Run all.** Cell 3 asks to mount Google Drive: allow it. Results are
   saved there as each run finishes, and if Colab disconnects, **Run all** again
   skips everything already done.

Only the model weights come from the internet (Hugging Face). The eval data and
scorer are the files you upload.

## 1. Configuration

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao"])

In [ ]:
# name, Hugging Face id, file slug, approx GPU memory needed in bfloat16 (GB)
MODELS = [
    ("Llama 3.2 3B",  "meta-llama/Llama-3.2-3B-Instruct",   "llama3.2_3b",    8),
    ("Qwen3-4B",      "Qwen/Qwen3-4B",                      "qwen3_4b",      10),
    ("Gemma 3 4B",    "google/gemma-3-4b-it",               "gemma3_4b",     11),
    ("Mistral 7B",    "mistralai/Mistral-7B-Instruct-v0.3", "mistral_7b",    17),
    ("Qwen3-8B",      "Qwen/Qwen3-8B",                      "qwen3_8b",      19),
    ("Gemma 3 12B",   "google/gemma-3-12b-it",              "gemma3_12b",    27),
    ("Qwen3-14B",     "Qwen/Qwen3-14B",                     "qwen3_14b",     32),
    ("Phi-4 14B",     "microsoft/phi-4",                    "phi4_14b",      32),
    ("Qwen3-30B-A3B", "Qwen/Qwen3-30B-A3B",                 "qwen3_30b_a3b", 64),
    ("Gemma 4 E2B",   "google/gemma-4-E2B-it",              "gemma4_e2b",    12),
]
RUN_ONLY = ["gemma4_e2b"]   # None = all.
                                       # Finished models are skipped anyway (resume).

EVALS = [           # run in this order for every model
    ("v1",       "teenpattibench_eval.jsonl",          ""),
    ("shuffled", "teenpattibench_eval_shuffled.jsonl", "_shuffled"),
]

# One precision for the whole batch, so all 18 runs are comparable with each other.
# bfloat16 is native on H100/A100 and avoids Gemma's float16 overflow. NOTE: the
# earlier Kaggle runs of Llama/Qwen3-4B/Mistral/Qwen3-8B were float16 on a T4, so
# their numbers here may differ slightly from those files.
DTYPE          = "bfloat16"
BATCH_SIZE     = 8
MAX_NEW_TOKENS = 16
DO_SAMPLE      = False
THINKING       = False     # Qwen3 thinking mode off, as in the earlier Qwen runs
SMOKE_N        = 5         # items tried first; a model that answers none is skipped
LIMIT          = None      # e.g. 20 for a fast end-to-end trial of the whole batch

DATA_DIR       = "/content"
MOUNT_DRIVE    = True      # strongly recommended for a multi-hour batch
DRIVE_DIR      = "/content/drive/MyDrive/teenpattibench_results"
LOCAL_DIR      = "/content/teenpattibench_results"
DELETE_WEIGHTS = True      # delete each model's download once done (needed for disk space)

## 2. Packages and GPU

In [ ]:
import importlib.metadata as md, subprocess, sys
from packaging import version

# Qwen3 / Qwen3-MoE need transformers >= 4.51; Gemma 3 >= 4.50; Gemma 4 needs >= 5.5.
_run = [m for m in MODELS if not RUN_ONLY or m[2] in RUN_ONLY]
NEED = "5.5.0" if any("gemma-4" in m[1] for m in _run) else "4.51.0"
tv = md.version("transformers") if any(d.metadata["Name"] == "transformers"
                                       for d in md.distributions()) else "0"
if version.parse(tv) < version.parse(NEED):
    print(f"transformers {tv} too old (need >= {NEED}) -- upgrading")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                           f"transformers>={NEED}", "accelerate"])
    print("UPGRADED. Now do Runtime -> Restart session, then Runtime -> Run all.")
    raise SystemExit("Restart the session so the new transformers is loaded.")
else:
    print(f"transformers {tv} OK")

import torch, transformers, shutil
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> A100.")
TORCH_DTYPE = getattr(torch, DTYPE)
GPUS = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
disk = shutil.disk_usage("/content")
print(f"transformers {transformers.__version__} | torch {torch.__version__}")
print(f"GPU {GPUS} | {VRAM_GB:.0f} GB | dtype {TORCH_DTYPE} | "
      f"disk free {disk.free / 1e9:.0f} GB")
if TORCH_DTYPE == torch.bfloat16 and not torch.cuda.is_bf16_supported():
    raise SystemExit("This GPU has no bfloat16. Use an H100 or A100.")

## 3. Output folder (Google Drive, so a disconnect loses nothing)

In [ ]:
import os
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = DRIVE_DIR
else:
    OUT_DIR = LOCAL_DIR
os.makedirs(OUT_DIR, exist_ok=True)
done = sorted(f for f in os.listdir(OUT_DIR) if f.startswith("preds_"))
print(f"results -> {OUT_DIR}")
print(f"already finished: {len(done)} prediction files" + (" (will be skipped)" if done else ""))

## 4. Load the uploaded files

In [ ]:
import json
need = [f for _, f, _ in EVALS] + ["teenpattibench_system_prompt.txt", "metrics.py"]
missing = [f for f in need if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    raise SystemExit(f"Missing from {DATA_DIR}: {missing}\nUpload them with the Files "
                     "panel and re-run this cell. If one appears as 'name (1).ext', "
                     "rename it or delete the duplicate.")
os.chdir(DATA_DIR)
sys.path.insert(0, DATA_DIR)
import metrics

SYSTEM_PATH = os.path.join(DATA_DIR, "teenpattibench_system_prompt.txt")
SYSTEM_PROMPT = open(SYSTEM_PATH, encoding="utf-8").read()
EVAL_ITEMS = {}
for tag, fname, _ in EVALS:
    items = [json.loads(l) for l in open(os.path.join(DATA_DIR, fname), encoding="utf-8")
             if l.strip()]
    if not LIMIT and len(items) != 2000:
        raise SystemExit(f"{fname}: expected 2,000 items, found {len(items):,}")
    EVAL_ITEMS[tag] = items[:LIMIT] if LIMIT else items
    print(f"{tag:<9} {fname:<38} {len(EVAL_ITEMS[tag]):,} items")
if [i["id"] for i in EVAL_ITEMS["v1"]] != [i["id"] for i in EVAL_ITEMS["shuffled"]]:
    raise SystemExit("v1 and shuffled eval files do not have the same ids in the same order.")
print(f"system prompt {len(SYSTEM_PROMPT)} chars")

## 5. Hugging Face login

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in with the HF_TOKEN secret.")
except Exception:
    login()

## 6. Pre-flight: access, memory and disk, for every model

Downloads only each model's small `config.json`. A model you haven't accepted the licence for,
or one too big for this GPU, is reported now and skipped later, so it can't stop the
batch halfway through.

In [ ]:
from huggingface_hub import hf_hub_download, HfApi

PLAN, api = [], HfApi()
print(f"{'model':<15} {'access':<9} {'needs':>6}  {'download':>9}  status")
for name, mid, slug, gb in MODELS:
    if RUN_ONLY and slug not in RUN_ONLY:
        continue
    status, size = "run", None
    try:
        hf_hub_download(mid, "config.json")
        access = "ok"
    except Exception as e:
        access, status = "DENIED", f"skip: no access ({type(e).__name__}) -- accept the licence"
    try:
        info = api.model_info(mid, files_metadata=True)
        size = sum((s.size or 0) for s in info.siblings
                   if s.rfilename.endswith((".safetensors", ".bin"))) / 1e9
    except Exception:
        pass
    if status == "run" and gb > VRAM_GB * 0.92:
        status = f"skip: needs ~{gb} GB, GPU has {VRAM_GB:.0f} GB"
    if all(os.path.exists(os.path.join(OUT_DIR, f"preds_{slug}_colab{suf}.jsonl"))
           for _, _, suf in EVALS) and status == "run":
        status = "done already"
    PLAN.append((name, mid, slug, gb, status))
    print(f"{name:<15} {access:<9} {gb:>4} GB  {(f'{size:.0f} GB' if size else '?'):>9}  {status}")

to_run = [p for p in PLAN if p[4] == "run"]
print(f"\n{len(to_run)} model(s) to run, {sum(p[4].startswith('skip') for p in PLAN)} skipped.")
if not to_run and not any(p[4] == "done already" for p in PLAN):
    raise SystemExit("Nothing can run. Fix the problems listed above.")

## 7. Inference helpers

The same logic as the single-model notebook, with one addition: `enable_thinking=False` is
passed to the chat template. Qwen3 reads it, as in the earlier Qwen runs, and every other
model's template ignores it.

In [ ]:
import re, gc, time, importlib, datetime
from transformers import AutoTokenizer
from tqdm.auto import tqdm

ACTION_WORDS = ("stay-blind", "see", "chaal", "raise", "pack", "show")
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56.0") \
           else "torch_dtype"

def extract_answer(completion):
    """First action word, matched against the FULL vocabulary (not the legal set)."""
    text = completion.strip().lower()
    hits = [(m.start(), w) for w in ACTION_WORDS
            for m in re.finditer(rf"(?<![a-z-]){re.escape(w)}(?![a-z-])", text)]
    return min(hits)[1] if hits else text

def load(mid):
    tok = AutoTokenizer.from_pretrained(mid)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model, err = None, []
    for cls_name in ("AutoModelForCausalLM", "AutoModelForMultimodalLM",
                     "Gemma4ForConditionalGeneration", "Gemma3ForConditionalGeneration"):
        try:
            cls = getattr(importlib.import_module("transformers"), cls_name, None)
            if cls is None:
                continue                     # class not in this transformers version
            model = cls.from_pretrained(mid, device_map={"": 0}, **{DTYPE_KW: TORCH_DTYPE})
            break
        except Exception as e:
            err.append(f"{cls_name}: {type(e).__name__}: {str(e)[:100]}")
    if model is None:
        raise RuntimeError("could not load: " + " | ".join(err))
    if model.dtype != TORCH_DTYPE:
        raise RuntimeError(f"loaded as {model.dtype}, expected {TORCH_DTYPE}")
    model.eval()
    gc_ = model.generation_config
    gc_.do_sample = DO_SAMPLE
    for k in ("temperature", "top_p", "top_k"):
        if hasattr(gc_, k):
            setattr(gc_, k, None)
    return tok, model

def make_prompt_builder(tok):
    def msgs(u, fold):
        return ([{"role": "user", "content": SYSTEM_PROMPT + "\n\n" + u}] if fold else
                [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": u}])
    def render(u, fold):
        return tok.apply_chat_template(msgs(u, fold), tokenize=False,
                                       add_generation_prompt=True, enable_thinking=THINKING)
    try:
        render("probe", False); fold = False
    except Exception:
        fold = True
    return (lambda u: render(u, fold)), fold

def make_generate(tok, model, build):
    @torch.no_grad()
    def generate(texts):
        enc = tok([build(t) for t in texts], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)     # template adds BOS itself
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE,
                             pad_token_id=tok.pad_token_id)
        return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return generate

def metadata(name, mid, tok, model, eval_path, n_items, pred_path, fold):
    meta = {
        "type": "metadata",
        "model": {"name": name, "model_id": mid,
                  "architecture": list(model.config.architectures or [type(model).__name__]),
                  "model_type": model.config.model_type,
                  "num_parameters": int(sum(p.numel() for p in model.parameters())),
                  "dtype": str(model.dtype), "device": str(model.device), "quantized": False},
        "tokenizer": {"name_or_path": tok.name_or_path, "vocab_size": tok.vocab_size,
                      "pad_token": tok.pad_token, "eos_token": tok.eos_token,
                      "bos_token": tok.bos_token},
        "generation": {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": DO_SAMPLE,
                       "temperature": None, "top_p": None, "top_k": None,
                       "thinking_enabled": THINKING},
        "dataset": {"name": "TeenPattiBench", "evaluation_file": eval_path,
                    "num_questions": n_items},
        "prompt": {"system_prompt_file": SYSTEM_PATH, "system_prompt": SYSTEM_PROMPT},
        "runtime": {"pytorch_version": torch.__version__,
                    "cuda_available": torch.cuda.is_available(),
                    "cuda_version": torch.version.cuda,
                    "gpu_count": torch.cuda.device_count(), "gpus": GPUS},
        "output": {"format": "JSONL", "prediction_file": pred_path},
    }
    if fold:
        meta["prompt"]["system_prompt_delivery"] = "folded_into_user_turn"
    return meta

def write_preds(path, meta, records):
    tmp = path + ".part"                     # atomic: a half-written file is never "done"
    with open(tmp, "w", encoding="utf-8") as fh:
        fh.write(json.dumps(meta) + "\n")
        for r in records:
            fh.write(json.dumps(r) + "\n")
    os.replace(tmp, path)

def score_and_save(items, pred_path, report_stem):
    recs = [json.loads(l) for l in open(pred_path, encoding="utf-8")][1:]
    preds = {r["id"]: {"action": r["answer"]} for r in recs}    # same remap as the repo
    rep = metrics.score(items, preds)
    text = metrics.format_report(rep, metrics.trivial_baselines(items))
    json.dump(rep, open(report_stem + ".json", "w"), indent=1)
    open(report_stem + ".txt", "w").write(text + "\n")
    return rep, text

def gpu_used_gb():
    return torch.cuda.memory_allocated(0) / 1e9

def delete_weights(mid):
    from huggingface_hub import scan_cache_dir
    cache = scan_cache_dir()
    revs = [r.commit_hash for repo in cache.repos if repo.repo_id == mid for r in repo.revisions]
    if revs:
        cache.delete_revisions(*revs).execute()
print("helpers ready")

## 8. Run the batch

In [ ]:
LOG = os.path.join(OUT_DIR, "run_log.jsonl")
def log(**kw):
    kw["time_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    with open(LOG, "a") as fh:
        fh.write(json.dumps(kw) + "\n")

t_all = time.time()
for idx, (name, mid, slug, gb, status) in enumerate(PLAN, 1):
    print("\n" + "=" * 78 + f"\n[{idx}/{len(PLAN)}] {name}  ({mid})  -- {status}\n" + "=" * 78)
    if status != "run":
        log(model=name, status=status)
        continue
    t0 = time.time()
    # Drop every reference to the previous model BEFORE loading the next one. A model
    # is only freed when nothing points at it -- `model`, `tok` and the `generate` /
    # `build` closures all do -- and loading a 14B model next to a leftover 12B one
    # does not fit in 40 GB.
    model = tok = generate = build = None
    gc.collect(); torch.cuda.empty_cache()
    print(f"GPU memory in use before loading: {gpu_used_gb():.1f} GB")
    try:
        tok, model = load(mid)
        build, fold = make_prompt_builder(tok)
        generate = make_generate(tok, model, build)
        print(f"loaded {type(model).__name__} in {time.time() - t0:.0f}s | "
              f"system prompt: {'FOLDED INTO USER TURN' if fold else 'system role'}")

        probe = EVAL_ITEMS["v1"][:SMOKE_N]
        got = [extract_answer(r) for r in generate([i["prompt"] for i in probe])]
        print("smoke test:", got)
        if not any(a in ACTION_WORDS for a in got):
            raise RuntimeError(f"smoke test: 0/{SMOKE_N} completions contained an action word")

        for tag, fname, suf in EVALS:
            pred_path = os.path.join(OUT_DIR, f"preds_{slug}_colab{suf}.jsonl")
            items = EVAL_ITEMS[tag]
            if os.path.exists(pred_path):
                print(f"  {tag}: already done, skipping")
                continue
            recs, t1 = [], time.time()
            for s in tqdm(range(0, len(items), BATCH_SIZE), desc=f"{slug} {tag}"):
                batch = items[s:s + BATCH_SIZE]
                for it, raw in zip(batch, generate([i["prompt"] for i in batch])):
                    recs.append({"id": it["id"], "answer": extract_answer(raw)})
            write_preds(pred_path, metadata(name, mid, tok, model,
                        os.path.join(DATA_DIR, fname), len(items), pred_path, fold), recs)
            rep, _ = score_and_save(items, pred_path,
                                    os.path.join(OUT_DIR, f"report_{slug}_colab{suf}"))
            aa = 100 * rep["overall"]["aa"] / rep["overall"]["n"]
            print(f"  {tag}: {len(recs):,} answers in {(time.time() - t1) / 60:.1f} min "
                  f"-> Action Accuracy {aa:.2f}%")
            log(model=name, eval=tag, status="done", action_accuracy=aa,
                minutes=round((time.time() - t1) / 60, 2), folded_system=fold)
    except torch.cuda.OutOfMemoryError as e:
        print(f"OUT OF MEMORY -- skipping {name}. Lower BATCH_SIZE or use a bigger GPU.")
        log(model=name, status="failed: out of memory")
    except Exception as e:
        print(f"FAILED -- skipping {name}: {type(e).__name__}: {e}")
        log(model=name, status=f"failed: {type(e).__name__}: {str(e)[:200]}")
    finally:
        model = tok = generate = build = None       # release this model's GPU memory
        gc.collect(); torch.cuda.empty_cache()
        if DELETE_WEIGHTS:
            try:
                delete_weights(mid)
            except Exception as e:
                print(f"  (could not delete cached weights: {e})")
        print(f"{name}: {(time.time() - t0) / 60:.1f} min | disk free "
              f"{shutil.disk_usage('/content').free / 1e9:.0f} GB")

print(f"\nBATCH FINISHED in {(time.time() - t_all) / 60:.1f} min")

## 9. Summary across all models

Every model × eval that has a prediction file, including ones finished in an earlier
session. For each stratum it shows the model's score next to what answering one word
forever would get there (the only fair comparison), plus the paired v1 → shuffled
change on the same 2,000 questions.

In [ ]:
import collections, csv

ALL = ("chaal", "pack", "raise", "show", "see", "stay-blind")
def best_const(sub):
    a = max(ALL, key=lambda x: sum(1 for i in sub if i["correct_action"] == x))
    return 100 * sum(1 for i in sub if i["correct_action"] == a) / len(sub)
def strata(items):
    g = [i for i in items if not i["is_mixed"]]
    return {"all": g,
            "seen": [i for i in g if i["meta"]["family"] == "seen_bet"],
            "conversion": [i for i in g if i["meta"]["family"] == "look"],
            "blind_bet": [i for i in g if i["meta"]["family"] == "blind_bet"]}
BASE = {k: best_const(v) for k, v in strata(EVAL_ITEMS["v1"]).items()}

rows, answers = [], {}
# Every model in MODELS, not just this session's RUN_ONLY subset -- otherwise a run of
# one model would overwrite the summary with a one-row table.
for name, mid, slug, gb in MODELS:
    per = {}
    for tag, fname, suf in EVALS:
        p = os.path.join(OUT_DIR, f"preds_{slug}_colab{suf}.jsonl")
        if not os.path.exists(p):
            continue
        L = [json.loads(l) for l in open(p, encoding="utf-8")]
        pr = {r["id"]: r["answer"] for r in L[1:]}
        S = strata(EVAL_ITEMS[tag])
        acc = {k: 100 * sum(1 for i in v if pr[i["id"]] == i["correct_action"]) / len(v)
               for k, v in S.items() if v}
        look = S["conversion"]
        by_turn = {t: 100 * sum(1 for i in look if i["meta"]["turn_index"] == t
                                and pr[i["id"]] == i["correct_action"])
                      / max(1, sum(1 for i in look if i["meta"]["turn_index"] == t))
                   for t in (1, 2, 3, 4)}
        per[tag] = pr
        answers[f"{slug}_{tag}"] = dict(collections.Counter(pr.values()).most_common())
        rows.append({"model": name, "eval": tag, "dtype": L[0]["model"]["dtype"],
                     **{f"aa_{k}": round(v, 2) for k, v in acc.items()},
                     **{f"conv_turn{t}": round(v, 2) for t, v in by_turn.items()},
                     "folded_system": "system_prompt_delivery" in L[0]["prompt"]})
    if len(per) == 2:
        g = strata(EVAL_ITEMS["v1"])["all"]
        right = {t: {i["id"] for i in g if per[t][i["id"]] == i["correct_action"]} for t in per}
        rows.append({"model": name, "eval": "shuffled - v1",
                     "aa_all": round(100 * (len(right["shuffled"]) - len(right["v1"])) / len(g), 2),
                     "fixed_by_shuffle": len(right["shuffled"] - right["v1"]),
                     "broken_by_shuffle": len(right["v1"] - right["shuffled"]),
                     "same_answer_pct": round(100 * sum(1 for k in per["v1"]
                                              if per["v1"][k] == per["shuffled"][k]) / len(per["v1"]), 1)})

print(f"one-word-forever baselines: all {BASE['all']:.2f}%  seen {BASE['seen']:.2f}%  "
      f"conversion {BASE['conversion']:.2f}%  blind_bet {BASE['blind_bet']:.2f}%\n")
print(f"{'model':<15}{'eval':<14}{'all':>8}{'seen':>8}{'conv':>8}{'blind':>8}   conversion by turn 1-4")
for r in rows:
    if r["eval"] == "shuffled - v1":
        print(f"{'':<15}{'Δ shuffled':<14}{r['aa_all']:>+8.2f}   fixed {r['fixed_by_shuffle']}, "
              f"broken {r['broken_by_shuffle']}, same answer {r['same_answer_pct']}%")
    else:
        t = "  ".join(f"{r[f'conv_turn{k}']:5.1f}" for k in (1, 2, 3, 4))
        flag = "  (system folded)" if r["folded_system"] else ""
        print(f"{r['model']:<15}{r['eval']:<14}{r['aa_all']:>7.2f}%{r.get('aa_seen', 0):>7.2f}%"
              f"{r.get('aa_conversion', 0):>7.2f}%{r.get('aa_blind_bet', 0):>7.2f}%   {t}{flag}")

keys = sorted({k for r in rows for k in r}, key=lambda k: (k not in ("model", "eval"), k))
with open(os.path.join(OUT_DIR, "summary.csv"), "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(rows)
json.dump({"baselines": BASE, "rows": rows, "answer_distributions": answers,
           "plan": [{"model": p[0], "id": p[1], "status": p[4]} for p in PLAN]},
          open(os.path.join(OUT_DIR, "summary.json"), "w"), indent=1)
print(f"\nsaved summary.csv and summary.json to {OUT_DIR}")
print("NOTE: a conversion score of exactly 62.96% means 'always stay-blind', and one that")
print("simply equals the key's see-rate per turn means 'always see' -- neither is learned timing.")

## 10. Download everything as one zip

In [ ]:
import zipfile
zpath = "/content/teenpattibench_results.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)):
        if f.endswith((".jsonl", ".json", ".txt", ".csv")) and not f.endswith(".part"):
            z.write(os.path.join(OUT_DIR, f), f)
with zipfile.ZipFile(zpath) as z:
    names = z.namelist()
print(f"{len(names)} files zipped:")
for n in names:
    print("  ", n)
from google.colab import files
files.download(zpath)